In [ ]:
#initialize
from risks import calculate_market_risk, calculate_life_risk, calculate_hslt_risk, calculate_cpty_type1, calculate_cdr_type_2_risk
from utils import input, output

import os
from dotenv import load_dotenv
from supabase import create_client, Client
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import openpyxl
from pathlib import Path
import random

# Initialize Supabase client
load_dotenv()
supabase: Client = create_client(
    os.environ.get("SUPABASE_URL"),
    os.environ.get("SUPABASE_KEY")
)

output_path = Path("/workspaces/SolvMate/outputs")

run_id = random.randint(1, 1000000)

In [ ]:
# Load input data 
data_id_enriched_market = input.run_Import("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xls",['MarketR'])

In [ ]:
#calculate market risk

#calculate interest rate risk
aggregation_tree_market_ir_enriched = calculate_market_risk.calculate_interest_rate_risk(data_id_enriched_market)
#calculate equity risk
aggregation_tree_market_eq_enriched = calculate_market_risk.calculate_equity_risk(data_id_enriched_market)
#calculate property risk
aggregation_tree_market_pro_enriched = calculate_market_risk.calculate_property_risk(data_id_enriched_market)
#calculate spread risk
aggregation_tree_market_spr_enriched = calculate_market_risk.calculate_spread_risk(data_id_enriched_market)
#calculate market risk
aggregation_tree_market_enriched = calculate_market_risk.calculate_total_market_risk(data_id_enriched_market)

In [4]:
aggregation_tree_market_all_risks = pd.concat([aggregation_tree_market_enriched,
                                                aggregation_tree_market_ir_enriched,
                                                aggregation_tree_market_eq_enriched,
                                                aggregation_tree_market_pro_enriched,
                                                aggregation_tree_market_spr_enriched], ignore_index=True)

In [ ]:
output.fill_templates_from_dataframe(aggregation_tree_market_all_risks)

In [ ]:

# Save intermediate DataFrames to a separate Excel file for debugging
debug_output_path = output_path / f"Debug_{run_id}.xlsx"
with pd.ExcelWriter(debug_output_path, engine='openpyxl') as debug_writer:
    data_id_enriched_market.to_excel(debug_writer, sheet_name='data_id_enriched_market', index=False)
    aggregation_tree_market_ir_enriched.to_excel(debug_writer, sheet_name='aggregation_tree_market_ir', index=False)
    aggregation_tree_market_eq_enriched.to_excel(debug_writer, sheet_name='aggregation_tree_market_eq', index=False)
    aggregation_tree_market_pro_enriched.to_excel(debug_writer, sheet_name='aggregation_tree_market_pro', index=False)
    aggregation_tree_market_spr_enriched.to_excel(debug_writer, sheet_name='aggregation_tree_market_spr', index=False)
    aggregation_tree_market_enriched.to_excel(debug_writer, sheet_name='aggregation_tree_market', index=False)


print(f"Debug file saved to: {debug_output_path}") 


In [ ]:
#LIFE RISK
data_id_enriched_life = input.run_Import("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xls",['LnH SLT UW'])




In [10]:
aggregation_tree_life_mor_enriched = calculate_life_risk.calculate_life_mortality_risk(data_id_enriched_life)
aggregation_tree_life_lon_enriched = calculate_life_risk.calculate_life_longevity_risk(data_id_enriched_life)
aggregation_tree_life_dis_enriched = calculate_life_risk.calculate_life_disability_risk(data_id_enriched_life)
aggregation_tree_life_lap_enriched = calculate_life_risk.calculate_life_lapse_risk(data_id_enriched_life)
aggregation_tree_life_exp_enriched = calculate_life_risk.calculate_life_expense_risk(data_id_enriched_life)
aggregation_tree_life_rev_enriched = calculate_life_risk.calculate_life_revision_risk(data_id_enriched_life)
aggregation_tree_life_cat_enriched = calculate_life_risk.calculate_life_catastrophe_risk(data_id_enriched_life)

In [11]:
aggregation_tree_life_all = pd.concat([aggregation_tree_life_mor_enriched,
                                        aggregation_tree_life_lon_enriched,
                                        aggregation_tree_life_dis_enriched,
                                        aggregation_tree_life_lap_enriched,
                                        aggregation_tree_life_exp_enriched,
                                        aggregation_tree_life_rev_enriched,
                                        aggregation_tree_life_cat_enriched], ignore_index=True)

In [ ]:

output.fill_templates_from_dataframe(aggregation_tree_life_all) 

In [ ]:
# CDR 
# note: not done yet
data_id_enriched_cdr = input.run_Import("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xls",['CDR'])
data_id_enriched_cdr_type2 = calculate_cdr_type_2_risk.read_exposures_table("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xlsb")

In [ ]:
# HSLT
# note: not done yet
data_id_enriched_hslt = input.run_Import("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xls",['LnH SLT UW'])

In [9]:
aggregation_tree_hslt_mor_enriched = calculate_hslt_risk.calculate_hslt_mortality_risk(data_id_enriched_hslt)
aggregation_tree_hslt_lon_enriched = calculate_hslt_risk.calculate_hslt_longevity_risk(data_id_enriched_hslt)
aggregation_tree_hslt_dis_enriched = calculate_hslt_risk.calculate_hslt_disability_risk(data_id_enriched_hslt)
aggregation_tree_hslt_lap_enriched = calculate_hslt_risk.calculate_hslt_lapse_risk(data_id_enriched_hslt)
aggregation_tree_hslt_exp_enriched = calculate_hslt_risk.calculate_hslt_expense_risk(data_id_enriched_hslt)
aggregation_tree_hslt_rev_enriched = calculate_hslt_risk.calculate_hslt_revision_risk(data_id_enriched_hslt)

/workspaces/SolvMate/src/utils/aggregation_tree.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Input insufficient' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  filtered_tree.at[index, "VALUE"] = calculate_value(


In [ ]:
from risks import calculate_cpty_type1
from utils import output

input_file = "/workspaces/SolvMate/input/04.12_SAS_Input_CDR.xlsb"
output_folder = "/workspaces/SolvMate/outputs"
    
cpd_aggregation_tree_enriched = calculate_cpty_type1.calculate_cpty_type1(input_file, output_folder)

output.fill_QRT_from_dataframe(cpd_aggregation_tree_enriched)